In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# STEP 1: LOAD
# ─────────────────────────────────────────────
df_raw = pd.read_csv('student.csv')  # change filename if needed
print(f"Raw shape: {df_raw.shape}")

# ─────────────────────────────────────────────
# STEP 2: DATA CLEANING
# ─────────────────────────────────────────────

df = df_raw.copy()

# 2a. Remove duplicates
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} duplicate rows → {len(df)} rows remain")

# 2b. Normalize Category (fix case: 'food' → 'Food', 'transport' → 'Transport')
df['Category'] = df['Category'].str.strip().str.title()
print(f"Unique categories after normalization: {df['Category'].unique()}")

# 2c. Fill missing Category using Merchant Name
merchant_cat_map = {
    'pg_rent': 'Utilities', 'electricity': 'Utilities', 'water': 'Utilities',
    'zomato': 'Food', 'swiggy': 'Food', 'subway': 'Food', 'ccd': 'Food',
    'mess_canteen': 'Food', 'local_dhaba': 'Food', 'dominos': 'Food',
    'uber': 'Transport', 'ola': 'Transport', 'rapido': 'Transport',
    'steam': 'Entertainment', 'netflix': 'Entertainment', 'spotify': 'Entertainment',
    'coursera': 'Education', 'udemy': 'Education',
}

def infer_category(row):
    if pd.isna(row['Category']):
        merchant_lower = str(row['Merchant_Name']).lower()
        for key, cat in merchant_cat_map.items():
            if key in merchant_lower:
                return cat
        return 'Miscellaneous'
    return row['Category']

null_before = df['Category'].isna().sum()
df['Category'] = df.apply(infer_category, axis=1)
print(f"Filled {null_before} missing Category values via merchant lookup")

# 2d. Parse Date — handles both dd-mm-yyyy and yyyy/mm/dd
def parse_date(d):
    for fmt in ('%d-%m-%Y', '%Y/%m/%d'):
        try:
            return pd.to_datetime(d, format=fmt)
        except:
            pass
    return pd.NaT

df['Date'] = df['Date'].apply(parse_date)
df = df.dropna(subset=['Date'])

# 2e. Cap outliers at 1st–99th percentile
p99 = df['Amount'].quantile(0.99)
p01 = df['Amount'].quantile(0.01)
df['Amount'] = df['Amount'].clip(lower=p01, upper=p99)
print(f"Outliers capped between ₹{p01:.0f} and ₹{p99:.0f}")

# 2f. Derive time columns
df['Month']     = df['Date'].dt.to_period('M').astype(str)
df['DayOfWeek'] = df['Date'].dt.day_name()
df['Week']      = df['Date'].dt.isocalendar().week.astype(int)
df['MonthNum']  = df['Date'].dt.month

print(f"\nCleaned shape: {df.shape}")

# Save cleaned CSV
df.to_csv('cleaned_transactions.csv', index=False)
print("Cleaned CSV saved as cleaned_transactions.csv")

Raw shape: (1530, 5)
Removed 20 duplicate rows → 1510 rows remain
Unique categories after normalization: ['Utilities' 'Entertainment' 'Food' 'Transport' 'Education' nan]
Filled 61 missing Category values via merchant lookup
Outliers capped between ₹52 and ₹1645

Cleaned shape: (1510, 9)
Cleaned CSV saved as cleaned_transactions.csv
